### Lab 1.3: Multi-Class Linear Classifier

In this lab you will explore multi-class classification and evaluate model generalization using a [dataset for heart disease prediction from the UCI ML repository](https://archive.ics.uci.edu/dataset/45/heart+disease).

In [6]:
!pip install -q -r https://raw.githubusercontent.com/calpoly-data4620/DATA-4620-Labs/refs/heads/main/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 28.6 MB/s eta 0:00:00


This ``ucimlrepo`` package provides a nice interface for accessing their datasets.

In [7]:
import numpy as np
from ucimlrepo import fetch_ucirepo

# fetch dataset
heart_disease = fetch_ucirepo(id=45)

# data (as pandas dataframes)
X = heart_disease.data.features
y = heart_disease.data.targets

# variable information
heart_disease.variables


,name,role,type,demographic,description,units,missing_values
0,age,Feature,Integer,Age,None,years,no
1,sex,Feature,Categorical,Sex,None,None,no
2,cp,Feature,Categorical,None,None,None,no
3,trestbps,Feature,Integer,None,resting blood pressure (on admission to the ho...,mm Hg,no
4,chol,Feature,Integer,None,serum cholestoral,mg/dl,no
5,fbs,Feature,Categorical,None,fasting blood sugar > 120 mg/dl,None,no
6,restecg,Feature,Categorical,None,None,None,no
7,thalach,Feature,Integer,None,maximum heart rate achieved,None,no
8,exang,Feature,Categorical,None,exercise induced angina,None,no
9,oldpeak,Feature,Integer,None,ST depression induced by exercise relative to ...,None,no


Here I remove the missing values from the features and labels.

In [8]:
bad = X.isna().any(axis=1)
X = X[~bad]
y = y[~bad]

Finally I convert the DataFrames to numpy arrays.

In [9]:
X = X.values
y = y.values.flatten()

The classification target is a number from 0-4 indicating the severity of heart disease.  Let's try fitting a linear model.

In [10]:
import sklearn

In [11]:
model = sklearn.linear_model.LogisticRegression().fit(X,y)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [12]:
model.score(X,y)

0.6094276094276094

### Exercises

1. Compute the $\mathbf{z}$ values for the classifier manually, i.e. compute

$$\mathbf{z}_i = \mathbf{W}\mathbf{x}_i+\mathbf{b}$$

for each data point $\mathbf{x}_i$.

Stack the resulting $\mathbf{z}_i$ vectors into a $N \times 5$ matrix $\mathbf{Z}$ where $N$ is the number of data points.

Do this first with a `for` loop.  Then do it without a `for` loop, instead using matrix multiplication and broadcasting.

*Hints*:
- Use `.shape` to get the shape of a Numpy matrix.
- ``@`` is the matrix multiplication operator in Numpy.
- For the version without a for loop, you will need to use the matrix transpose which is `.T` in Numpy.

In [13]:
N = X.shape[0]
D = X.shape[1]
W = model.coef_
b = model.intercept_
Z_loop = np.zeros((N, 5))

for i in range(N):
  xi = X[i]
  zi = W @ xi + b
  Z_loop[i] = zi

Z_loop

array([[ 1.02965934,  0.44582056, -0.31957023, -0.33764968, -0.81825999],
       [-0.20348864,  0.06117336,  0.22315353,  0.39084486, -0.47168311],
       [-1.27329015,  0.49374443,  0.50158279,  0.36805567, -0.09009274],
       ...,
       [-0.77803946,  0.37040162,  0.24355905,  0.43611131, -0.27203252],
       [-1.11471072,  0.53175285,  0.16408259,  0.64466206, -0.22578678],
       [ 3.6584551 ,  0.56252979, -1.13347767, -1.59792916, -1.48957806]])

In [14]:
Z_loop2 = np.zeros((N, 5))
Z_loop2 = W @ X.T + b.reshape(-1, 1)
Z_loop2 = Z_loop2.T
Z_loop2

array([[ 1.02965934,  0.44582056, -0.31957023, -0.33764968, -0.81825999],
       [-0.20348864,  0.06117336,  0.22315353,  0.39084486, -0.47168311],
       [-1.27329015,  0.49374443,  0.50158279,  0.36805567, -0.09009274],
       ...,
       [-0.77803946,  0.37040162,  0.24355905,  0.43611131, -0.27203252],
       [-1.11471072,  0.53175285,  0.16408259,  0.64466206, -0.22578678],
       [ 3.6584551 ,  0.56252979, -1.13347767, -1.59792916, -1.48957806]])

Print out the $\mathbf{z}$ values for the first example in the dataset and the first label.   Determine if the classifier is correctly classifying the first example in the dataset.

In [15]:
z_first = Z_loop[0]
print(z_first)

[ 1.02965934  0.44582056 -0.31957023 -0.33764968 -0.81825999]


In [16]:
print(f"Correct label is: {y[0]}")

Correct label is: 0


In [17]:
predicted_label = np.argmax(z_first)
print(f"Predicted label is: {predicted_label}")

Predicted label is: 0


The predicted label and predicted label both match with 0.

2. Use ``sklearn.model_selection.train_test_split`` to split ``X`` and ``y`` into 90% train and 10% test splits.  Note that this should be done in a single call to ``train_test_split``.

*Note*: Pass ``random_state=1234`` to ``train_test_split`` to ensure you get the same result from random shuffling each time.


In [18]:
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    X, y, test_size=0.1, random_state=1234
)

Fit the model to the training split and calculate accuracy on the test split.  How does it compare to the previous accuracy value (when the model was trained and evaluated on the same data)?

In [19]:
model =  sklearn.linear_model.LogisticRegression().fit(X_train, y_train)
model.score(X_test, y_test)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.5

The accuracy went down from about 0.6 to 0.5 from the previous accuracy of the full dataset being used as training data and then used as the test data as well.

3. Run $k$-fold cross validation with $k=5$ and interpret the results (see `sklearn.model_selection.cross_val_score`).

In [22]:
scores = sklearn.model_selection.cross_val_score(sklearn.linear_model.LogisticRegression(), X_train, y_train, cv = 5)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

In [23]:
print(scores)
print(f"Mean accuracy: {scores.mean():.4f}")
print(f"Std deviation: {scores.std():.4f}")

[0.55555556 0.59259259 0.66037736 0.56603774 0.60377358]
Mean accuracy: 0.5957
Std deviation: 0.0367


These are the accuracies across each cross fold validation. So the first fold had an accuracy of 0.555..., the second fold had an accuracy of 0.660..., etc. The average accuracy was 0.5957, with a standard deviation between the different folds' accuracies of 0.0367. This means the accuracy was pretty consistent across all the folds with an average of 0.5957.